# ABRIR DOCKER DESKTOP Y CONTAINER BLAST NCBI

In [1]:
import subprocess

result = subprocess.run(
    ["docker", "run", "--rm", "ncbi/blast", "blastn", "-version"],
    capture_output=True,
    text=True
)

print(result.stdout)

blastn: 2.17.0+
 Package: blast 2.17.0, build Jul 29 2025 18:15:49



# CREAR BASE DE REFERENCIA 

In [2]:
from pathlib import Path
import subprocess

reference_fasta = Path(
    r"C:\Users\berna\Desktop\PAPER METADIV\V1_7_7_RUNS\MetaDiv_Builder_V1_7_7\output\ITS\For_R\only_fungi\species_only_p1_sppn08\sequences.fasta"
)

blast_db_dir = Path(
    r"C:\Users\berna\Desktop\PAPER METADIV\V1_7_7_RUNS\MetaDiv_Builder_V1_7_7\BLAST\blast_db"
)

blast_db_dir.mkdir(exist_ok=True)

cmd = [
    "docker", "run", "--rm",
    "-v", f"{reference_fasta.parent}:/data",
    "-v", f"{blast_db_dir}:/blast_db",
    "ncbi/blast",
    "makeblastdb",
    "-in", f"/data/{reference_fasta.name}",
    "-dbtype", "nucl",
    "-out", "/blast_db/metadiv_db"
]

subprocess.run(cmd, check=True)

CompletedProcess(args=['docker', 'run', '--rm', '-v', 'C:\\Users\\berna\\Desktop\\PAPER METADIV\\V1_7_7_RUNS\\MetaDiv_Builder_V1_7_7\\output\\ITS\\For_R\\only_fungi\\species_only_p1_sppn08:/data', '-v', 'C:\\Users\\berna\\Desktop\\PAPER METADIV\\V1_7_7_RUNS\\MetaDiv_Builder_V1_7_7\\BLAST\\blast_db:/blast_db', 'ncbi/blast', 'makeblastdb', '-in', '/data/sequences.fasta', '-dbtype', 'nucl', '-out', '/blast_db/metadiv_db'], returncode=0)

# CORRER BLASTN

In [3]:
from pathlib import Path
import subprocess
import pandas as pd

# Paths
project_dir = Path(r"C:\Users\berna\Desktop\PAPER METADIV\V1_7_7_RUNS\MetaDiv_Builder_V1_7_7")
blast_dir = project_dir / "BLAST"

query_dir = blast_dir / "query"
db_dir = blast_dir / "blast_db"
results_dir = blast_dir / "blast_results"

results_dir.mkdir(parents=True, exist_ok=True)

# Detect query FASTA automatically
query_files = list(query_dir.glob("*.fasta")) + list(query_dir.glob("*.fa"))

if len(query_files) == 0:
    raise FileNotFoundError("No FASTA file found in BLAST/query")

if len(query_files) > 1:
    raise ValueError("More than one FASTA file found in BLAST/query. Keep only one.")

query_fasta = query_files[0]

# Output
blast_tsv = results_dir / "blast_results.tsv"

# Run BLAST
cmd = [
    "docker", "run", "--rm",
    "-v", f"{query_dir}:/query",
    "-v", f"{db_dir}:/blast_db",
    "-v", f"{results_dir}:/results",
    "ncbi/blast",
    "blastn",
    "-query", f"/query/{query_fasta.name}",
    "-db", "/blast_db/metadiv_db",
    "-out", "/results/blast_results.tsv",
    "-outfmt",
    "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovs",
    "-evalue", "1e-20",
    "-num_threads", "8",
    "-max_target_seqs", "10"
]

print("Running BLAST...")
subprocess.run(cmd, check=True)
print("BLAST completed.")

# Load results
cols = [
    "qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
    "qstart", "qend", "sstart", "send", "evalue", "bitscore", "qcovs"
]

blast_df = pd.read_csv(
    blast_tsv,
    sep="\t",
    header=None,
    names=cols
)

# Save all hits
blast_df.to_csv(results_dir / "blast_all_hits.csv", index=False)

# Best hit per query
best_hits = (
    blast_df
    .sort_values(
        by=["qseqid", "bitscore", "qcovs", "pident", "evalue"],
        ascending=[True, False, False, False, True]
    )
    .drop_duplicates("qseqid")
    .reset_index(drop=True)
)

best_hits.to_csv(results_dir / "blast_best_hits.csv", index=False)

print("Total hits:", len(blast_df))
print("Queries with hits:", best_hits["qseqid"].nunique())
print("Results saved in:", results_dir)

best_hits

Running BLAST...
BLAST completed.
Total hits: 30
Queries with hits: 3
Results saved in: C:\Users\berna\Desktop\PAPER METADIV\V1_7_7_RUNS\MetaDiv_Builder_V1_7_7\BLAST\blast_results


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qcovs
0,Li111_cfA,Lactarius_0066,94.788,614,28,2,4,615,655,44,0.0,953,99
1,Li37_ss,Lactarius_0117,96.491,627,19,2,1,624,663,37,0.0,1038,100
2,Li86_cfB,Lactarius_0066,93.841,617,34,2,1,615,658,44,0.0,928,100
